# BÀI THỰC HÀNH: PHÂN CỤM DỮ LIỆU VÀ CÁC THUẬT TOÁN CLUSTERING
**Học phần: Phân tích Dữ liệu với Python (Data Analysis with Python)**  
**Giảng viên: TS. Vũ Đức Minh – Khoa Khoa học dữ liệu và Trí tuệ nhân tạo, NEU**  
**Thời gian cập nhật: 22 tháng 9 năm 2026**

---

## Mục tiêu Học tập

Sau khi hoàn thành bài thực hành này, học viên có khả năng:
1. **Nắm vững bản chất toán học:** Hiểu rõ cơ chế hội tụ của K-Means, các liên kết của Hierarchical, nguyên lý mật độ của DBSCAN/OPTICS, và tối ưu cực đại kỳ vọng (EM) trong GMM.
2. **Thực hiện thuần thục tính toán tay (Hand Calculations):** Tự mình tính toán từng bước khoảng cách, cập nhật tâm, xây dựng ma trận liên kết, phân loại điểm mật độ và ma trận trách nhiệm xác suất.
3. **Thành thạo thư viện Scikit-Learn & SciPy:** Sử dụng thành thạo `KMeans`, `AgglomerativeClustering`, `DBSCAN`, `OPTICS`, `GaussianMixture` và các công cụ trực quan hoá.
4. **Đánh giá mô hình toàn diện:** Áp dụng chuẩn xác các chỉ số nội bộ (*Silhouette Coefficient, Davies-Bouldin, Calinski-Harabasz*) và tiêu chuẩn thông tin (*BIC, AIC*).
5. **Tự kiểm tra kiến thức:** Giải quyết các bài tập dạng điền vào chỗ trống (`### TODO: ...`) được kiểm chứng tự động bằng hệ thống `assert test`.

---

## Hướng dẫn Thực hành
- Đọc kỹ phần tóm tắt lý thuyết và các công thức toán học trước mỗi phần.
- Quan sát kỹ từng bước của các **Bài tập tính tay chi tiết**, sau đó chạy code đối chiếu để kiểm tra kết quả số học.
- Ở các ô **Bài tập điền khuyết**, hãy hoàn thiện phần code còn thiếu tại vị trí `### TODO: ...` và chạy ô kiểm tra `assert` tương ứng để chấm điểm tự động.



## 0. Chuẩn bị Môi trường & Thư viện

Trong bài thực hành này, chúng ta sử dụng các thư viện phân tích dữ liệu và học máy tiêu chuẩn trong hệ sinh thái Python:
- `numpy`: Thao tác mảng số học, đại số tuyến tính, tính toán khoảng cách vector.
- `pandas`: Cấu trúc dữ liệu DataFrame, quản lý bảng số liệu khách hàng.
- `matplotlib.pyplot` & `seaborn`: Trực quan hóa dữ liệu 2D, biểu đồ phân cụm, elip hiệp phương sai.
- `scipy.cluster.hierarchy` & `scipy.spatial.distance`: Xây dựng ma trận khoảng cách và vẽ biểu đồ hình cây Dendrogram.
- `sklearn.cluster`, `sklearn.mixture`, `sklearn.metrics`: Các thuật toán phân cụm chuẩn công nghiệp và bộ chỉ số thẩm định chất lượng.



In [ ]:
# Import các thư viện cần thiết
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Thư viện Scipy cho phân tích khoảng cách và phân cụm thứ bậc
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Thư viện Scikit-Learn cho phân cụm, chuẩn hóa và đánh giá
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, OPTICS
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_blobs, make_moons

# Cấu hình thẩm mỹ hiển thị
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# Cố định seed ngẫu nhiên để tái lập kết quả thí nghiệm
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment setup completed successfully!")


## 1. Tóm tắt Lý thuyết & Cheatsheet Thư viện Scikit-Learn

| Thuật toán | Lớp phương pháp | Lớp đối tượng Scikit-Learn / SciPy | Tham số quan trọng cốt lõi | Thuộc tính sau khi Fit (`.fit()`) | Ưu điểm & Nhược điểm chính |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **K-Means / K-Means++** | Centroid-based | `sklearn.cluster.KMeans` | `n_clusters`, `init='k-means++'`, `n_init=10`, `max_iter=300` | `.cluster_centers_`, `.labels_`, `.inertia_`, `.n_iter_` | **(+)** Nhanh $O(NKI)$, mở rộng tốt.<br>**(-)** Cần biết $K$, nhạy cảm outlier, chỉ tìm cụm cầu. |
| **Hierarchical Agglomerative** | Connectivity | `scipy.cluster.hierarchy.linkage`<br>`sklearn.cluster.AgglomerativeClustering` | `n_clusters`, `metric='euclidean'`, `linkage='ward'/'complete'/'single'` | `.labels_`, `.children_`, `.n_leaves_` | **(+)** Trực quan qua Dendrogram, không cần chọn trước $K$.<br>**(-)** Tốn bộ nhớ $O(N^2)$, tính toán chậm $O(N^3)$. |
| **DBSCAN** | Density-based | `sklearn.cluster.DBSCAN` | `eps`, `min_samples`, `metric='euclidean'` | `.labels_` (nhãn `-1` là nhiễu), `.core_sample_indices_` | **(+)** Cụm hình dạng tùy ý, tự lọc outlier.<br>**(-)** Nhạy cảm khi chọn `eps`, kém trên cụm đa mật độ. |
| **OPTICS** | Density-based | `sklearn.cluster.OPTICS` | `min_samples`, `max_eps=np.inf`, `cluster_method='xi'`, `xi=0.05` | `.labels_`, `.reachability_`, `.core_distances_`, `.ordering_` | **(+)** Xử lý xuất sắc cụm đa mật độ, Reachability Plot.<br>**(-)** Chậm hơn DBSCAN, nhiều siêu tham số. |
| **GMM (EM Algorithm)** | Distribution-based | `sklearn.mixture.GaussianMixture` | `n_components`, `covariance_type='full'/'diag'`, `max_iter=100` | `.means_`, `.covariances_`, `.weights_`, `.converged_`, `.bic()`, `.aic()` | **(+)** Phân cụm mềm xác suất, hình dạng Elip co giãn linh hoạt.<br>**(-)** Cực tiểu cục bộ, dễ suy biến ma trận hiệp phương sai. |

---
### Các Chỉ số Đánh giá Chất lượng Phân cụm (Validation Metrics)

1. **Silhouette Coefficient ($s \in [-1, 1]$):**  
   $$s(i) = rac{b(i) - a(i)}{\max(a(i), b(i))}$$
   - $a(i)$: Khoảng cách trung bình từ mẫu $i$ đến các mẫu khác trong cùng cụm.
   - $b(i)$: Khoảng cách trung bình nhỏ nhất từ mẫu $i$ đến các mẫu thuộc cụm khác gần nhất.
   - Điểm số: $s > 0.5 \implies$ Cụm rất chặt chẽ và tách biệt rõ; $s pprox 0 \implies$ Vùng ranh giới giao thoa; $s < 0 \implies$ Gán nhầm cụm.
   
2. **Davies-Bouldin Index (DBI $\ge 0$):**  
   $$DB = rac{1}{K}\sum_{i=1}^K \max_{j 
eq i} \left( rac{s_i + s_j}{d(\mu_i, \mu_j)} ight)$$
   - Tỷ số giữa độ phân tán nội cụm ($s_i + s_j$) và khoảng cách giữa các tâm $d(\mu_i, \mu_j)$. **Càng nhỏ càng tốt** (cụm càng đặc và cách xa nhau).

3. **Calinski-Harabasz Index (Variance Ratio Criterion):**  
   $$CH = rac{	ext{SSB} / (K - 1)}{	ext{SSW} / (N - K)}$$
   - Tỷ số giữa độ biến thiên liên cụm (Between-cluster dispersion) và nội cụm (Within-cluster dispersion). **Càng lớn càng tốt**.

4. **Bayesian Information Criterion (BIC) & Akaike Information Criterion (AIC):**  
   $$	ext{BIC} = -2\ln \hat{L} + p \ln N, \quad 	ext{AIC} = -2\ln \hat{L} + 2p$$
   - Dùng cho mô hình xác suất GMM. Phạt số lượng tham số $p$ để tránh overfitting. **Mô hình có BIC/AIC nhỏ nhất là tối ưu**.



## 2. Các Bài tập Tính Tay Chi Tiết Từng Bước (Step-by-Step Hand Calculations)

Trong phần này, chúng ta đi sâu vào bản chất toán học của từng thuật toán qua các ví dụ số học thực tế có thể tính toán chi tiết bằng bút và giấy. Sau đó, ta sẽ viết code Python đối chiếu để xác nhận tính chính xác.

---

### Bài toán 1: Vòng lặp Thuật toán K-Means (1D/2D Hand Calculation)

**Đề bài:**  
Cho tập dữ liệu gồm $N = 6$ điểm trong không gian 2 chiều $\mathbb{R}^2$:
- $P_1 = (1, 1)$
- $P_2 = (1.5, 2)$
- $P_3 = (3, 4)$
- $P_4 = (5, 7)$
- $P_5 = (3.5, 5)$
- $P_6 = (4.5, 5)$

Yêu cầu gom thành $K = 2$ cụm. Giả sử tại bước khởi tạo ($t = 0$), hai tâm cụm được chọn ngẫu nhiên là:
$$\mu_1^{(0)} = P_1 = (1, 1), \quad \mu_2^{(0)} = P_4 = (5, 7)$$

Thực hiện **trọn vẹn 1 vòng lặp** (Epoch 1) của thuật toán Lloyd:
1. Tính bình phương khoảng cách Euclidean $d^2(P_i, \mu_k) = (x_{i1} - \mu_{k1})^2 + (x_{i2} - \mu_{k2})^2$ từ mỗi điểm đến hai tâm.
2. Gán mỗi điểm vào cụm gần hơn: $c^{(i)} = rg\min_k d^2(P_i, \mu_k)$.
3. Tính toạ độ trọng tâm mới $\mu_1^{(1)}$ và $\mu_2^{(1)}$.
4. Tính tổng bình phương khoảng cách nội cụm (Inertia / WCSS) trước và sau khi cập nhật tâm.



#### Lời giải Chi tiết Bài toán 1:

**Bước 1 & 2: Bảng tính khoảng cách và gán cụm:**

| Điểm $P_i$ | Toạ độ $(x_1, x_2)$ | $d^2(P_i, \mu_1)$ với $\mu_1=(1,1)$ | $d^2(P_i, \mu_2)$ với $\mu_2=(5,7)$ | Cụm được gán |
| :---: | :---: | :---: | :---: | :---: |
| **$P_1$** | $(1, 1)$ | $(1-1)^2 + (1-1)^2 = \mathbf{0.0}$ | $(1-5)^2 + (1-7)^2 = 16 + 36 = 52.0$ | **Cụm 1** |
| **$P_2$** | $(1.5, 2)$ | $(1.5-1)^2 + (2-1)^2 = 0.25 + 1 = \mathbf{1.25}$ | $(1.5-5)^2 + (2-7)^2 = 12.25 + 25 = 37.25$ | **Cụm 1** |
| **$P_3$** | $(3, 4)$ | $(3-1)^2 + (4-1)^2 = 4 + 9 = \mathbf{13.0}$ | $(3-5)^2 + (4-7)^2 = 4 + 9 = \mathbf{13.0}$ | **Cụm 1** (hòa, gán cụm 1) |
| **$P_4$** | $(5, 7)$ | $(5-1)^2 + (7-1)^2 = 16 + 36 = 52.0$ | $(5-5)^2 + (7-7)^2 = \mathbf{0.0}$ | **Cụm 2** |
| **$P_5$** | $(3.5, 5)$ | $(3.5-1)^2 + (5-1)^2 = 6.25 + 16 = 22.25$ | $(3.5-5)^2 + (5-7)^2 = 2.25 + 4 = \mathbf{6.25}$ | **Cụm 2** |
| **$P_6$** | $(4.5, 5)$ | $(4.5-1)^2 + (5-1)^2 = 12.25 + 16 = 28.25$ | $(4.5-5)^2 + (5-7)^2 = 0.25 + 4 = \mathbf{4.25}$ | **Cụm 2** |

Kết quả phân cụm sau bước gán:
- $C_1 = \{P_1, P_2, P_3\}$ (gồm 3 điểm)
- $C_2 = \{P_4, P_5, P_6\}$ (gồm 3 điểm)

---

**Bước 3: Cập nhật toạ độ tâm cụm mới $\mu_k^{(1)}$:**
- Với Cụm 1:
  $$\mu_1^{(1)} = rac{1}{3} \left( P_1 + P_2 + P_3 ight) = \left( rac{1 + 1.5 + 3}{3}, rac{1 + 2 + 4}{3} ight) = \left( rac{5.5}{3}, rac{7}{3} ight) pprox \mathbf{(1.8333, 2.3333)}$$
- Với Cụm 2:
  $$\mu_2^{(1)} = rac{1}{3} \left( P_4 + P_5 + P_6 ight) = \left( rac{5 + 3.5 + 4.5}{3}, rac{7 + 5 + 5}{3} ight) = \left( rac{13.0}{3}, rac{17.0}{3} ight) pprox \mathbf{(4.3333, 5.6667)}$$

---

**Bước 4: Tính giá trị WCSS (Inertia) giảm sau cập nhật:**
- Trước khi cập nhật tâm (khoảng cách đến $\mu^{(0)}$):
  $$	ext{WCSS}^{(0)} = (0.0 + 1.25 + 13.0) + (0.0 + 6.25 + 4.25) = 14.25 + 10.50 = \mathbf{24.75}$$
- Sau khi cập nhật tâm mới (khoảng cách đến $\mu^{(1)}$):
  - Nội cụm 1: $(1-1.833)^2+(1-2.333)^2 + (1.5-1.833)^2+(2-2.333)^2 + (3-1.833)^2+(4-2.333)^2 = 2.4722 + 0.2222 + 4.1389 = \mathbf{6.8333}$
  - Nội cụm 2: $(5-4.333)^2+(7-5.667)^2 + (3.5-4.333)^2+(5-5.667)^2 + (4.5-4.333)^2+(5-5.667)^2 = 2.2222 + 1.1389 + 0.4722 = \mathbf{3.8333}$
  - $	ext{WCSS}^{(1)} = 6.8333 + 3.8333 = \mathbf{10.6667}$

$$\Delta 	ext{WCSS} = 24.75 - 10.6667 = \mathbf{14.0833} \quad (	ext{Hàm mục tiêu giảm mạnh!})$$



In [ ]:
# Code Python kiểm chứng bài tính tay K-Means
X_demo = np.array([
    [1.0, 1.0],
    [1.5, 2.0],
    [3.0, 4.0],
    [5.0, 7.0],
    [3.5, 5.0],
    [4.5, 5.0]
])

# Khởi tạo tâm ban đầu đúng như bài tính tay
initial_centers = np.array([
    [1.0, 1.0],
    [5.0, 7.0]
])

# Tính khoảng cách Euclidean bình phương tới 2 tâm
dists_sq = np.sum((X_demo[:, np.newaxis, :] - initial_centers[np.newaxis, :, :]) ** 2, axis=2)
cluster_assigned = np.argmin(dists_sq, axis=1)

# Cập nhật tâm mới
new_center_1 = X_demo[cluster_assigned == 0].mean(axis=0)
new_center_2 = X_demo[cluster_assigned == 1].mean(axis=0)

wcss_0 = np.sum(dists_sq[np.arange(len(X_demo)), cluster_assigned])

new_centers = np.vstack([new_center_1, new_center_2])
new_dists_sq = np.sum((X_demo[:, np.newaxis, :] - new_centers[np.newaxis, :, :]) ** 2, axis=2)
wcss_1 = np.sum(new_dists_sq[np.arange(len(X_demo)), cluster_assigned])

print("--- KẾT QUẢ ĐỐI CHỨNG PYTHON ---")
for i, pt in enumerate(X_demo):
    print(f"Điểm P{i+1}: d^2(mu_1)={dists_sq[i, 0]:.2f}, d^2(mu_2)={dists_sq[i, 1]:.2f} => Cụm {cluster_assigned[i]+1}")

print(f"\nTâm cụm 1 mới: {new_center_1} (Lý thuyết: [1.8333, 2.3333])")
print(f"Tâm cụm 2 mới: {new_center_2} (Lý thuyết: [4.3333, 5.6667])")
print(f"WCSS trước cập nhật: {wcss_0:.4f} (Lý thuyết: 24.7500)")
print(f"WCSS sau cập nhật:   {wcss_1:.4f} (Lý thuyết: 10.6667)")

# Tự động assert kiểm tra khớp tuyệt đối
assert np.allclose(new_center_1, [5.5/3, 7.0/3])
assert np.allclose(new_center_2, [13.0/3, 17.0/3])
assert np.isclose(wcss_0, 24.75)
assert np.isclose(wcss_1, 10.6666667)
print("=> KHỚP HOÀN TOÀN 100% VỚI BÀI TÍNH TAY!")


---

### Bài toán 2: Phân cụm Phân cấp Thứ bậc (Hierarchical Clustering Hand Calculation)

**Đề bài:**  
Cho ma trận khoảng cách đối xứng $D_0$ kích thước $5 	imes 5$ giữa 5 đối tượng $\{A, B, C, D, E\}$:

$$D_0 = egin{matrix}
 & A & B & C & D & E \
A & 0 & 2 & 6 & 10 & 9 \
B & 2 & 0 & 5 & 9 & 8 \
C & 6 & 5 & 0 & 4 & 5 \
D & 10 & 9 & 4 & 0 & 3 \
E & 9 & 8 & 5 & 3 & 0
\end{matrix}$$

Thực hiện quá trình sáp nhập phân cụm thứ bậc gom cụm theo phương pháp:
1. **Single Linkage** (Khoảng cách cực tiểu: $D(C_1, C_2) = \min_{x \in C_1, y \in C_2} d(x, y)$).
2. **Complete Linkage** (Khoảng cách cực đại: $D(C_1, C_2) = \max_{x \in C_1, y \in C_2} d(x, y)$).

---

#### Lời giải Chi tiết Bài toán 2:

##### Quá trình 1: Single Linkage

- **Bước 1:** Khoảng cách nhỏ nhất trong $D_0$ là $d(A, B) = \mathbf{2}$.  
  Ta sáp nhập $A$ và $B$ thành cụm mới $(AB)$ tại ngưỡng khoảng cách **$h_1 = 2$**.  
  Cập nhật ma trận khoảng cách $D_1$ với các cụm $\{(AB), C, D, E\}$ bằng công thức cực tiểu:
  - $d((AB), C) = \min(d(A,C), d(B,C)) = \min(6, 5) = \mathbf{5}$
  - $d((AB), D) = \min(d(A,D), d(B,D)) = \min(10, 9) = \mathbf{9}$
  - $d((AB), E) = \min(d(A,E), d(B,E)) = \min(9, 8) = \mathbf{8}$

  $$D_1 = egin{matrix}
   & (AB) & C & D & E \
  (AB) & 0 & 5 & 9 & 8 \
  C & 5 & 0 & 4 & 5 \
  D & 9 & 4 & 0 & 3 \
  E & 8 & 5 & 3 & 0
  \end{matrix}$$

- **Bước 2:** Khoảng cách nhỏ nhất trong $D_1$ là $d(D, E) = \mathbf{3}$.  
  Sáp nhập $D$ và $E$ thành cụm $(DE)$ tại độ cao **$h_2 = 3$**.  
  Cập nhật khoảng cách:
  - $d((AB), (DE)) = \min(d(AB,D), d(AB,E)) = \min(9, 8) = \mathbf{8}$
  - $d(C, (DE)) = \min(d(C,D), d(C,E)) = \min(4, 5) = \mathbf{4}$

  $$D_2 = egin{matrix}
   & (AB) & C & (DE) \
  (AB) & 0 & 5 & 8 \
  C & 5 & 0 & 4 \
  (DE) & 8 & 4 & 0
  \end{matrix}$$

- **Bước 3:** Khoảng cách nhỏ nhất trong $D_2$ là $d(C, (DE)) = \mathbf{4}$.  
  Sáp nhập $C$ và $(DE)$ thành cụm $(CDE)$ tại độ cao **$h_3 = 4$**.  
  Khoảng cách còn lại:
  - $d((AB), (CDE)) = \min(d(AB,C), d(AB,DE)) = \min(5, 8) = \mathbf{5}$

- **Bước 4:** Sáp nhập hai cụm cuối cùng $(AB)$ và $(CDE)$ tại độ cao **$h_4 = 5$**.  
  **Thứ tự sáp nhập Single Linkage:** $(A, B) @ 2 	o (D, E) @ 3 	o (C, (DE)) @ 4 	o ((AB), (CDE)) @ 5$.

---

##### Quá trình 2: Complete Linkage (Cực đại)

- **Bước 1:** Khoảng cách nhỏ nhất ban đầu vẫn là $d(A, B) = \mathbf{2}$. Sáp nhập $(AB)$ tại $h_1 = 2$.  
  Cập nhật khoảng cách theo cực đại:
  - $d((AB), C) = \max(d(A,C), d(B,C)) = \max(6, 5) = \mathbf{6}$
  - $d((AB), D) = \max(d(A,D), d(B,D)) = \max(10, 9) = \mathbf{10}$
  - $d((AB), E) = \max(d(A,E), d(B,E)) = \max(9, 8) = \mathbf{9}$

- **Bước 2:** Khoảng cách nhỏ nhất tiếp theo là $d(D, E) = \mathbf{3}$. Sáp nhập $(DE)$ tại $h_2 = 3$.  
  Cập nhật:
  - $d((AB), (DE)) = \max(d(AB,D), d(AB,E)) = \max(10, 9) = \mathbf{10}$
  - $d(C, (DE)) = \max(d(C,D), d(C,E)) = \max(4, 5) = \mathbf{5}$

  $$D_2^{complete} = egin{matrix}
   & (AB) & C & (DE) \
  (AB) & 0 & 6 & 10 \
  C & 6 & 0 & 5 \
  (DE) & 10 & 5 & 0
  \end{matrix}$$

- **Bước 3:** Khoảng cách nhỏ nhất trong $D_2^{complete}$ là $d(C, (DE)) = \mathbf{5}$.  
  Sáp nhập thành cụm $(CDE)$ tại độ cao **$h_3 = 5$**.  
  Khoảng cách cuối cùng:
  - $d((AB), (CDE)) = \max(d(AB,C), d(AB,DE)) = \max(6, 10) = \mathbf{10}$

- **Bước 4:** Sáp nhập hai cụm cuối cùng tại độ cao **$h_4 = 10$**.



In [ ]:
# Code Python kiểm chứng bài tính tay Hierarchical Clustering bằng SciPy
D_matrix = np.array([
    [0, 2, 6, 10, 9],
    [2, 0, 5, 9, 8],
    [6, 5, 0, 4, 5],
    [10, 9, 4, 0, 3],
    [9, 8, 5, 3, 0]
], dtype=float)

# Chuyển đổi ma trận đối xứng sang dạng condensed vector cho scipy linkage
condensed_D = squareform(D_matrix)
labels_pt = ['A', 'B', 'C', 'D', 'E']

# 1. Single Linkage
Z_single = linkage(condensed_D, method='single')
print("--- SCIPY SINGLE LINKAGE MATRIX [Cluster1, Cluster2, Distance, Count] ---")
print(Z_single)

# 2. Complete Linkage
Z_complete = linkage(condensed_D, method='complete')
print("\n--- SCIPY COMPLETE LINKAGE MATRIX [Cluster1, Cluster2, Distance, Count] ---")
print(Z_complete)

# Kiểm chứng các ngưỡng độ cao ghép cụm
assert np.allclose(Z_single[:, 2], [2.0, 3.0, 4.0, 5.0]), "Lỗi tính Single Linkage"
assert np.allclose(Z_complete[:, 2], [2.0, 3.0, 5.0, 10.0]), "Lỗi tính Complete Linkage"
print("\n=> TẤT CẢ CÁC BƯỚC GHÉP VÀ ĐỘ CAO KHỚP CHÍNH XÁC 100% VỚI BÀI TÍNH TAY!")


---

### Bài toán 3: Phân loại Điểm Mật độ DBSCAN (Core, Border, Noise & Clusters)

**Đề bài:**  
Cho tập gồm $N = 8$ điểm trong không gian 2 chiều $\mathbb{R}^2$:
- $P_1 = (2, 2)$
- $P_2 = (2, 3)$
- $P_3 = (3, 2)$
- $P_4 = (8, 8)$
- $P_5 = (8, 9)$
- $P_6 = (25, 25)$  *(điểm nằm rất xa)*
- $P_7 = (3, 3)$
- $P_8 = (8, 7)$

Cấu hình tham số DBSCAN:
- Bán kính láng giềng: $\epsilon = 1.5$
- Ngưỡng số lượng điểm tối thiểu: $	ext{MinPts} = 3$ (tính cả chính điểm đó).

**Yêu cầu:**
1. Liệt kê tập láng giềng $\epsilon$-neighborhood $N_\epsilon(P_i) = \{ P_j \mid d(P_i, P_j) \le \epsilon \}$.
2. Phân loại từng điểm là **Core**, **Border**, hay **Noise**.
3. Xác định các cụm được hình thành qua liên thông mật độ (Density-Connected).



#### Lời giải Chi tiết Bài toán 3:

Khoảng cách Euclidean giữa hai điểm: $d(P_i, P_j) = \sqrt{(x_{i1}-x_{j1})^2 + (x_{i2}-x_{j2})^2}$.  
Ta có: $\epsilon = 1.5 \implies \epsilon^2 = 2.25$. Một điểm $P_j$ thuộc $N_\epsilon(P_i) \iff d^2(P_i, P_j) \le 2.25$.

**Bước 1 & 2: Bảng khảo sát từng điểm:**

| Điểm $P_i$ | Toạ độ | Khoảng cách đến các điểm khác $\le 1.5$ | Tập láng giềng $N_\epsilon(P_i)$ | Số lượng $|N_\epsilon|$ | Phân loại |
| :---: | :---: | :--- | :---: | :---: | :---: |
| **$P_1$** | $(2, 2)$ | $d(P_1, P_2)=1$, $d(P_1, P_3)=1$, $d(P_1, P_7)=\sqrt{2}pprox 1.414$ | $\{P_1, P_2, P_3, P_7\}$ | **4** ($\ge 3$) | **Core Point** |
| **$P_2$** | $(2, 3)$ | $d(P_2, P_1)=1$, $d(P_2, P_7)=1$, $d(P_2, P_3)=\sqrt{2}pprox 1.414$ | $\{P_2, P_1, P_3, P_7\}$ | **4** ($\ge 3$) | **Core Point** |
| **$P_3$** | $(3, 2)$ | $d(P_3, P_1)=1$, $d(P_3, P_7)=1$, $d(P_3, P_2)=\sqrt{2}pprox 1.414$ | $\{P_3, P_1, P_2, P_7\}$ | **4** ($\ge 3$) | **Core Point** |
| **$P_7$** | $(3, 3)$ | $d(P_7, P_2)=1$, $d(P_7, P_3)=1$, $d(P_7, P_1)=\sqrt{2}pprox 1.414$ | $\{P_7, P_1, P_2, P_3\}$ | **4** ($\ge 3$) | **Core Point** |
| **$P_4$** | $(8, 8)$ | $d(P_4, P_5)=1$, $d(P_4, P_8)=1$ | $\{P_4, P_5, P_8\}$ | **3** ($\ge 3$) | **Core Point** |
| **$P_5$** | $(8, 9)$ | $d(P_5, P_4)=1$, $d(P_5, P_8)=2 > 1.5$ | $\{P_5, P_4\}$ | **2** ($< 3$) | **Border Point** (thuộc $N_\epsilon$ của Core $P_4$) |
| **$P_8$** | $(8, 7)$ | $d(P_8, P_4)=1$, $d(P_8, P_5)=2 > 1.5$ | $\{P_8, P_4\}$ | **2** ($< 3$) | **Border Point** (thuộc $N_\epsilon$ của Core $P_4$) |
| **$P_6$** | $(25, 25)$ | $d(P_6, P_k) \ge \sqrt{17^2+16^2} pprox 23.3 \gg 1.5$ | $\{P_6\}$ | **1** ($< 3$) | **Noise Point** (Không lân cận Core nào) |

---

**Bước 3: Lan truyền mật độ và hình thành cụm:**
- Nhóm 1: $\{P_1, P_2, P_3, P_7\}$ đều là Core Points và giao thoa láng giềng $\implies$ Tạo thành **Cụm 0 (Cluster 0)**: $\{P_1, P_2, P_3, P_7\}$.
- Nhóm 2: $P_4$ là Core Point, liên thông trực tiếp tới 2 Border Points $P_5, P_8$ $\implies$ Tạo thành **Cụm 1 (Cluster 1)**: $\{P_4, P_5, P_8\}$.
- Điểm cô lập: $P_6$ không thuộc bất kỳ láng giềng của Core nào $\implies$ Gán nhãn **Noise (Nhiễu, nhãn -1)**: $\{P_6\}$.



In [ ]:
# Code Python kiểm chứng bài tính tay DBSCAN
pts_dbscan = np.array([
    [2.0, 2.0],  # P1
    [2.0, 3.0],  # P2
    [3.0, 2.0],  # P3
    [8.0, 8.0],  # P4
    [8.0, 9.0],  # P5
    [25.0, 25.0],# P6
    [3.0, 3.0],  # P7
    [8.0, 7.0]   # P8
])

db = DBSCAN(eps=1.5, min_samples=3)
db_labels = db.fit_predict(pts_dbscan)
core_indices = set(db.core_sample_indices_)

print("--- KẾT QUẢ ĐỐI CHỨNG DBSCAN SCIKIT-LEARN ---")
for i, pt in enumerate(pts_dbscan):
    pt_type = "Core Point" if i in core_indices else ("Border Point" if db_labels[i] != -1 else "Noise Point")
    print(f"P{i+1} ({pt[0]:4.1f}, {pt[1]:4.1f}): Nhãn = {db_labels[i]:2d} | Phân loại = {pt_type}")

# Assert kiểm tra
assert core_indices == {0, 1, 2, 3, 6}, "Tập Core Points không khớp!"
assert db_labels[5] == -1, "P6 phải là điểm Nhiễu (Noise)!"
assert db_labels[0] == db_labels[1] == db_labels[2] == db_labels[6] == 0, "Cụm 1 không khớp!"
assert db_labels[3] == db_labels[4] == db_labels[7] == 1, "Cụm 2 không khớp!"
print("\n=> KẾT QUẢ PYTHON HOÀN TOÀN KHỚP 100% VỚI BÀI TÍNH TAY!")


---

### Bài toán 4: Thuật toán Kỳ vọng - Cực đại hóa (EM) cho Mô hình Trộn Gaussian (GMM Hand Calculation)

**Đề bài:**  
Cho tập dữ liệu 1 chiều gồm $N = 4$ điểm:
$$X = \{1.0, 2.0, 5.0, 6.0\}$$

Giả sử dữ liệu được tạo bởi $K = 2$ phân phối chuẩn 1 chiều với các tham số ban đầu:
- Cụm 1: $\pi_1 = 0.5, \quad \mu_1 = 1.5, \quad \sigma_1^2 = 1.0 \implies \sigma_1 = 1.0$
- Cụm 2: $\pi_2 = 0.5, \quad \mu_2 = 5.5, \quad \sigma_2^2 = 1.0 \implies \sigma_2 = 1.0$

Hàm mật độ xác suất chuẩn 1 chiều:
$$\mathcal{N}(x \mid \mu, \sigma^2) = rac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -rac{(x - \mu)^2}{2\sigma^2} ight)$$
Với $\sigma = 1.0$, ta có $rac{1}{\sqrt{2\pi}} pprox \mathbf{0.3989}$.

**Yêu cầu:**
1. **E-step:** Tính ma trận trách nhiệm $\gamma_{nk} = P(z_{nk}=1 \mid x_n) = rac{\pi_k \mathcal{N}(x_n \mid \mu_k, \sigma_k^2)}{\sum_{j=1}^2 \pi_j \mathcal{N}(x_n \mid \mu_j, \sigma_j^2)}$.
2. **M-step:** Cập nhật các tham số mới:
   - Tổng trách nhiệm: $N_k = \sum_{n=1}^N \gamma_{nk}$
   - Trọng số cụm: $\pi_k^{new} = rac{N_k}{N}$
   - Kỳ vọng mới: $\mu_k^{new} = rac{1}{N_k} \sum_{n=1}^N \gamma_{nk} x_n$
   - Phương sai mới: $(\sigma_k^2)^{new} = rac{1}{N_k} \sum_{n=1}^N \gamma_{nk} (x_n - \mu_k^{new})^2$



#### Lời giải Chi tiết Bài toán 4:

**Bước 1: E-step (Tính mật độ xác suất và trách nhiệm):**

Ta tính giá trị mật độ $\mathcal{N}_1(x_n)$ và $\mathcal{N}_2(x_n)$ cho 4 điểm:
- Với $x_1 = 1.0$:
  - $(x_1 - \mu_1)^2 = (1.0 - 1.5)^2 = 0.25 \implies \mathcal{N}_1(1.0) = 0.3989 \cdot e^{-0.25/2} = 0.3989 \cdot 0.8825 = \mathbf{0.3521}$
  - $(x_1 - \mu_2)^2 = (1.0 - 5.5)^2 = 20.25 \implies \mathcal{N}_2(1.0) = 0.3989 \cdot e^{-20.25/2} = 0.3989 \cdot 0.000039 = \mathbf{0.0000}$
  - Vì $\pi_1 = \pi_2 = 0.5$, ta có:
    $$\gamma_{11} = rac{0.3521}{0.3521 + 0.0000} pprox \mathbf{1.0000}, \quad \gamma_{12} pprox \mathbf{0.0000}$$

- Với $x_2 = 2.0$:
  - $(x_2 - \mu_1)^2 = (2.0 - 1.5)^2 = 0.25 \implies \mathcal{N}_1(2.0) = \mathbf{0.3521}$
  - $(x_2 - \mu_2)^2 = (2.0 - 5.5)^2 = 12.25 \implies \mathcal{N}_2(2.0) = 0.3989 \cdot e^{-12.25/2} = 0.3989 \cdot 0.0022 = \mathbf{0.0009}$
  - Trách nhiệm:
    $$\gamma_{21} = rac{0.3521}{0.3521 + 0.0009} = rac{0.3521}{0.3530} pprox \mathbf{0.9975}, \quad \gamma_{22} pprox \mathbf{0.0025}$$

- Tương tự do tính đối xứng hoàn hảo qua tâm $3.5$:
  - Với $x_3 = 5.0$: $\gamma_{31} pprox \mathbf{0.0025}, \quad \gamma_{32} pprox \mathbf{0.9975}$
  - Với $x_4 = 6.0$: $\gamma_{41} pprox \mathbf{0.0000}, \quad \gamma_{42} pprox \mathbf{1.0000}$

Bảng ma trận trách nhiệm $\Gamma$:

$$\Gamma = egin{bmatrix}
\gamma_{11} & \gamma_{12} \
\gamma_{21} & \gamma_{22} \
\gamma_{31} & \gamma_{32} \
\gamma_{41} & \gamma_{42}
\end{bmatrix} pprox egin{bmatrix}
1.0000 & 0.0000 \
0.9975 & 0.0025 \
0.0025 & 0.9975 \
0.0000 & 1.0000
\end{bmatrix}$$

---

**Bước 2: M-step (Cập nhật tham số):**

1. **Tổng trách nhiệm hiệu dụng $N_k$:**
   $$N_1 = 1.0000 + 0.9975 + 0.0025 + 0.0000 = \mathbf{2.0000}$$
   $$N_2 = 0.0000 + 0.0025 + 0.9975 + 1.0000 = \mathbf{2.0000}$$

2. **Trọng số mới $\pi_k^{new}$:**
   $$\pi_1^{new} = rac{2.0}{4} = \mathbf{0.5000}, \quad \pi_2^{new} = rac{2.0}{4} = \mathbf{0.5000}$$

3. **Kỳ vọng mới $\mu_k^{new}$:**
   $$\mu_1^{new} = rac{1}{2.0} \left( 1.0000 \cdot 1.0 + 0.9975 \cdot 2.0 + 0.0025 \cdot 5.0 + 0.0000 \cdot 6.0 ight) = rac{1.0 + 1.995 + 0.0125}{2.0} = rac{3.0075}{2.0} pprox \mathbf{1.50375}$$
   $$\mu_2^{new} = rac{1}{2.0} \left( 0.0000 \cdot 1.0 + 0.0025 \cdot 2.0 + 0.9975 \cdot 5.0 + 1.0000 \cdot 6.0 ight) = rac{0.005 + 4.9875 + 6.0}{2.0} = rac{10.9925}{2.0} pprox \mathbf{5.49625}$$

4. **Phương sai mới $(\sigma_k^2)^{new}$:**
   Tính tổng các đóng góp có trọng số từ cả 4 điểm:
   $$(\sigma_1^2)^{new} = rac{1}{2.0} \left[ 1.0 \cdot (1.0 - 1.5038)^2 + 0.9975 \cdot (2.0 - 1.5038)^2 + 0.0025 \cdot (5.0 - 1.5038)^2 + 0 ight]$$
   $$= rac{0.2538 + 0.2456 + 0.0306}{2.0} = rac{0.5300}{2.0} pprox \mathbf{0.2653}$$
   $$(\sigma_2^2)^{new} pprox \mathbf{0.2653}$$

*Nhận xét:* Sau 1 vòng lặp EM, phương sai giảm từ $1.0$ xuống xấp xỉ $0.2653$, các điểm $\{1, 2\}$ gom quanh tâm $1.5$ với độ lệch chuẩn $\sigma pprox \sqrt{0.2653} pprox 0.515$, phản ánh cực kỳ chuẩn xác cấu trúc phân cụm!



In [ ]:
# Code Python mô phỏng 1 vòng lặp EM để kiểm chứng bài tính tay GMM
from scipy.stats import norm

X_gmm = np.array([1.0, 2.0, 5.0, 6.0])
pi = np.array([0.5, 0.5])
mu = np.array([1.5, 5.5])
sigma_sq = np.array([1.0, 1.0])

# E-step: Tính ma trận trách nhiệm gamma
pdf_1 = norm.pdf(X_gmm, loc=mu[0], scale=np.sqrt(sigma_sq[0]))
pdf_2 = norm.pdf(X_gmm, loc=mu[1], scale=np.sqrt(sigma_sq[1]))

numerator = np.column_stack([pi[0] * pdf_1, pi[1] * pdf_2])
gamma = numerator / np.sum(numerator, axis=1, keepdims=True)

# M-step: Cập nhật
N_k = np.sum(gamma, axis=0)
pi_new = N_k / len(X_gmm)
mu_new = np.sum(gamma * X_gmm[:, np.newaxis], axis=0) / N_k
sigma_sq_new = np.sum(gamma * (X_gmm[:, np.newaxis] - mu_new[np.newaxis, :])**2, axis=0) / N_k

print("--- MA TRẬN TRÁCH NHIỆM GAMMA (E-STEP) ---")
print(np.round(gamma, 4))
print(f"\nTổng trách nhiệm N_k: {N_k}")
print(f"Trọng số mới pi_new: {pi_new}")
print(f"Kỳ vọng mới mu_new:   {mu_new} (Lý thuyết: [1.5038, 5.4962])")
print(f"Phương sai mới var:  {sigma_sq_new} (Lý thuyết: [0.2653, 0.2653])")

assert np.allclose(N_k, [2.0, 2.0], atol=1e-3)
assert np.allclose(mu_new, [1.50375, 5.49625], atol=1e-3)
assert np.allclose(sigma_sq_new, [0.2653, 0.2653], atol=1e-3)
print("\n=> TÍNH TOÁN EM KHỚP CHÍNH XÁC VỚI BÀI TOÁN TÍNH TAY!")


## 3. Trực Quan Hóa Minh Họa Lý Thuyết Chuyên Sâu

Dưới đây là 4 khối code hoàn chỉnh trực quan hóa các đặc trưng hình học cốt lõi của 4 trường phái thuật toán.



In [ ]:
# Trực quan hóa 1: Biên phân vùng Voronoi và quỹ đạo hội tụ của tâm K-Means
from scipy.spatial import Voronoi, voronoi_plot_2d

# Sinh dữ liệu 3 cụm hình cầu
X_blobs, y_blobs = make_blobs(n_samples=300, centers=[[-3, -2], [2, 4], [4, -3]], cluster_std=0.8, random_state=42)

# Khởi tạo tâm K-Means
kmeans_viz = KMeans(n_clusters=3, init='random', n_init=1, max_iter=1, random_state=12)
kmeans_viz.fit(X_blobs)
init_centers = kmeans_viz.cluster_centers_

kmeans_final = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
kmeans_final.fit(X_blobs)
final_centers = kmeans_final.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Đồ thị 1: Phân vùng Voronoi
vor = Voronoi(final_centers)
voronoi_plot_2d(vor, ax=axes[0], show_vertices=False, line_colors='crimson', line_width=2, line_style='--')
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_final.labels_, cmap='viridis', alpha=0.6, s=25)
axes[0].scatter(final_centers[:, 0], final_centers[:, 1], c='red', marker='X', s=200, edgecolors='black', label='Tâm cụm cuối')
axes[0].set_title("Biên phân vùng Voronoi (Voronoi Tessellation)", fontsize=13, fontweight='bold')
axes[0].legend()

# Đồ thị 2: Quỹ đạo dịch chuyển tâm
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c='gray', alpha=0.3, s=20)
axes[1].scatter(init_centers[:, 0], init_centers[:, 1], c='orange', marker='o', s=150, edgecolors='black', label='Tâm ban đầu (Epoch 0)')
axes[1].scatter(final_centers[:, 0], final_centers[:, 1], c='red', marker='X', s=200, edgecolors='black', label='Tâm hội tụ')
for c0, c1 in zip(init_centers, final_centers):
    axes[1].annotate('', xy=c1, xytext=c0, arrowprops=dict(facecolor='black', edgecolor='black', width=1.5, headwidth=8))
axes[1].set_title("Quỹ đạo Dịch chuyển Trọng tâm K-Means", fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Trực quan hóa 2: Cây Phân cấp Dendrogram và Ngưỡng Cắt Cụm
sample_idx = np.random.choice(len(X_blobs), size=30, replace=False)
X_sub = X_blobs[sample_idx]

# Tính ma trận liên kết Ward Linkage
Z_sub = linkage(X_sub, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z_sub, leaf_rotation=90, leaf_font_size=10, color_threshold=8.0)
plt.axhline(y=8.0, color='r', linestyle='--', linewidth=2, label='Ngưỡng cắt khoảng cách = 8.0 (Tạo 3 cụm)')
plt.title("Biểu đồ Phân cấp Thứ bậc (Dendrogram) với Ngưỡng Cắt Cụm", fontsize=13, fontweight='bold')
plt.xlabel("Chỉ số Mẫu dữ liệu (Sample Index)", fontsize=11)
plt.ylabel("Khoảng cách Ward Linkage", fontsize=11)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Trực quan hóa 3: Đồ thị K-Distance Elbow xác định Epsilon cho DBSCAN
# Sinh dữ liệu 2 vầng trăng khuyết có nhiễu
X_moons, _ = make_moons(n_samples=400, noise=0.08, random_state=42)

# Tính khoảng cách tới láng giềng thứ k (k = MinPts - 1 = 4)
k_neighbors = 4
nbrs = NearestNeighbors(n_neighbors=k_neighbors).fit(X_moons)
distances, indices = nbrs.kneighbors(X_moons)

# Sắp xếp khoảng cách tăng dần
sorted_distances = np.sort(distances[:, k_neighbors - 1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Đồ thị 1: K-Distance Elbow Plot
axes[0].plot(sorted_distances, color='navy', linewidth=2.5)
axes[0].axhline(y=0.18, color='crimson', linestyle='--', linewidth=2, label='Điểm khuỷu tay (Epsilon tối ưu = 0.18)')
axes[0].set_title(f"Đồ thị K-Distance Plot (k={k_neighbors})", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Các điểm sắp xếp theo khoảng cách tăng dần", fontsize=11)
axes[0].set_ylabel(f"Khoảng cách đến láng giềng thứ {k_neighbors}", fontsize=11)
axes[0].legend(fontsize=11)

# Đồ thị 2: Kết quả gom cụm DBSCAN
db_viz = DBSCAN(eps=0.18, min_samples=5).fit(X_moons)
colors = ['red' if l == -1 else plt.cm.Set1(l) for l in db_viz.labels_]
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=colors, s=30, alpha=0.8)
axes[1].set_title(f"Kết quả Phân cụm DBSCAN (eps=0.18, min_samples=5)", fontsize=13, fontweight='bold')
axes[1].text(0.05, 0.92, f"Số cụm tìm thấy: {len(set(db_viz.labels_)) - (1 if -1 in db_viz.labels_ else 0)}\nSố điểm nhiễu: {np.sum(db_viz.labels_ == -1)}", 
             transform=axes[1].transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()


In [ ]:
# Trực quan hóa 4: Elip Tin cậy của Mô hình Trộn Gaussian (GMM Confidence Ellipses)
from matplotlib.patches import Ellipse

def draw_ellipse(position, covariance, ax=None, **kwargs):
    ax = ax or plt.gca()
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)
    
    # Vẽ các mức tin cậy 1-sigma, 2-sigma, 3-sigma
    for k in [1, 2, 3]:
        ax.add_patch(Ellipse(position, k * width, k * height, angle=angle, **kwargs))

# Sinh dữ liệu hình elip nghiêng bất đối xứng
np.random.seed(42)
X_ellip_1 = np.random.randn(200, 2) @ [[1.5, 0.8], [0.2, 0.5]] + [2, 3]
X_ellip_2 = np.random.randn(200, 2) @ [[0.5, -0.9], [1.2, 0.3]] + [-2, -1]
X_ellip = np.vstack([X_ellip_1, X_ellip_2])

gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42).fit(X_ellip)

plt.figure(figsize=(8, 6))
plt.scatter(X_ellip[:, 0], X_ellip[:, 1], c=gmm.predict(X_ellip), cmap='viridis', s=25, alpha=0.5)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', marker='X', s=180, label='Tâm Gauss (Mu)')

for mean, cov in zip(gmm.means_, gmm.covariances_):
    draw_ellipse(mean, cov, alpha=0.2, facecolor='orange', edgecolor='red', linewidth=1.5)

plt.title("Elip Đồng Mức Tin cậy ($1\sigma, 2\sigma, 3\sigma$) của Mô hình GMM", fontsize=13, fontweight='bold')
plt.xlabel("Trục X1", fontsize=11)
plt.ylabel("Trục X2", fontsize=11)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 4. Bài tập Lập trình Điền Khuyết (Fill-in-the-Blank Code Exercises with Tests)

Trong phần này, bạn sẽ làm việc với một bộ dữ liệu khách hàng tổng hợp (Customer Mall Segmentation Dataset) chứa các thuộc tính mua sắm thực tế. Hãy hoàn thành các vị trí `### TODO: ...` để xây dựng quy trình phân tích phân cụm chuẩn khoa học.

### Chuẩn bị Bộ dữ liệu Thực hành
Bộ dữ liệu gồm 250 khách hàng với các trường thông tin:
- `Annual_Income_k$`: Thu nhập hàng năm (nghìn USD).
- `Spending_Score`: Điểm chi tiêu do trung tâm thương mại chấm điểm (thang 1 - 100).
- `Loyalty_Years`: Số năm gắn bó của khách hàng.
- `Purchase_Frequency`: Số lượt mua sắm mỗi tháng.



In [ ]:
# Tạo bộ dữ liệu khách hàng thực hành
np.random.seed(42)
n_cust = 250

# Tạo 4 phân khúc khách hàng tự nhiên
c1 = np.random.randn(60, 4) * [8, 8, 1.0, 1.5] + [30, 20, 1.5, 3]    # Thu nhập thấp, chi tiêu thấp
c2 = np.random.randn(70, 4) * [10, 9, 1.2, 2.0] + [35, 80, 2.0, 8]   # Thu nhập thấp, chi tiêu cao
c3 = np.random.randn(60, 4) * [9, 7, 1.5, 2.0] + [90, 25, 4.0, 4]    # Thu nhập cao, chi tiêu thấp (tiết kiệm)
c4 = np.random.randn(60, 4) * [12, 10, 1.8, 2.5] + [95, 85, 5.5, 12] # VIP: Thu nhập cao, chi tiêu cao

df_customers = pd.DataFrame(
    np.vstack([c1, c2, c3, c4]),
    columns=['Annual_Income_k$', 'Spending_Score', 'Loyalty_Years', 'Purchase_Frequency']
)

print(f"Kích thước dataset khách hàng: {df_customers.shape}")
df_customers.head()


---

### Lab 1: K-Means Clustering & Phương pháp Khuỷu tay (Elbow) + Silhouette

#### Nhiệm vụ của bạn:
1. Chuẩn hóa dữ liệu bằng `StandardScaler()`. Lưu ma trận đã chuẩn hóa vào biến `X_scaled`.
2. Chạy vòng lặp với số lượng cụm $k \in [2, 8]$:
   - Fit mô hình `KMeans` với `init='k-means++'`, `n_init=10`, `random_state=42`.
   - Lưu giá trị WCSS (`.inertia_`) vào danh sách `inertias`.
   - Tính hệ số `silhouette_score(X_scaled, kmeans.labels_)` và lưu vào danh sách `silhouette_scores`.
3. Khởi tạo mô hình K-Means tối ưu với $K = 4$ cụm và huấn luyện trên `X_scaled`. Lưu nhãn vào biến `kmeans_labels`.



In [ ]:
# BÀI TẬP LAB 1: ĐIỀN CODE VÀO CÁC CHỖ TRỐNG TODO

# Bước 1: Chuẩn hóa toàn bộ dữ liệu khách hàng
scaler = StandardScaler()
### TODO 1.1: Chuẩn hóa df_customers bằng fit_transform() và lưu vào X_scaled
X_scaled = scaler.fit_transform(df_customers)

# Bước 2: Vòng lặp khảo sát K từ 2 đến 8
k_range = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_range:
    ### TODO 1.2: Khởi tạo mô hình KMeans với n_clusters=k, init='k-means++', n_init=10, random_state=42
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    
    ### TODO 1.3: Fit mô hình km vào X_scaled
    km.fit(X_scaled)
    
    ### TODO 1.4: Lưu giá trị inertia_ vào danh sách inertias
    inertias.append(km.inertia_)
    
    ### TODO 1.5: Tính điểm silhouette_score và thêm vào danh sách silhouette_scores
    score = silhouette_score(X_scaled, km.labels_)
    silhouette_scores.append(score)

# Bước 3: Huấn luyện mô hình tối ưu với K=4
### TODO 1.6: Khởi tạo và fit_predict mô hình KMeans với n_clusters=4
best_kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
kmeans_labels = best_kmeans.fit_predict(X_scaled)

print("Đã hoàn thành Lab 1! Hãy chạy ô assert bên dưới để kiểm tra kết quả.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG CHO LAB 1 (ASSERT TEST)
assert X_scaled.shape == (250, 4), "Lỗi: X_scaled phải có kích thước (250, 4)!"
assert np.allclose(X_scaled.mean(axis=0), 0, atol=1e-2), "Lỗi: Dữ liệu chưa được chuẩn hóa trung bình về 0!"
assert len(inertias) == 7, "Lỗi: inertias phải chứa đúng 7 giá trị tương ứng k từ 2 đến 8!"
assert len(silhouette_scores) == 7, "Lỗi: silhouette_scores phải chứa đúng 7 giá trị!"
assert inertias[0] > inertias[-1], "Lỗi: Inertia phải giảm đơn điệu khi tăng k!"
assert len(set(kmeans_labels)) == 4, "Lỗi: kmeans_labels phải chứa đúng 4 nhãn cụm!"
assert np.max(silhouette_scores) > 0.4, "Lỗi: Điểm silhouette tối ưu chưa đạt yêu cầu!"

print("XUẤT SẮC! BẠN ĐÃ VƯỢT QUA TẤT CẢ CÁC TEST CASE CỦA LAB 1!")


---

### Lab 2: Phân cụm Phân cấp Thứ bậc (Hierarchical Agglomerative Clustering)

#### Nhiệm vụ của bạn:
1. Dùng SciPy hàm `linkage()` để tính ma trận liên kết với phương pháp `method='ward'` trên ma trận `X_scaled`. Lưu vào biến `Z_matrix`.
2. Dùng Scikit-Learn `AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')` để phân cụm dữ liệu `X_scaled`.
3. Lưu nhãn dự báo vào biến `hierarchical_labels`.
4. Tính chỉ số Davies-Bouldin Index (`davies_bouldin_score`) giữa `X_scaled` và `hierarchical_labels`. Lưu vào biến `dbi_hierarchical`.



In [ ]:
# BÀI TẬP LAB 2: ĐIỀN CODE VÀO CÁC CHỖ TRỐNG TODO

### TODO 2.1: Tính ma trận liên kết Z_matrix bằng linkage() của SciPy với method='ward'
Z_matrix = linkage(X_scaled, method='ward')

### TODO 2.2: Khởi tạo mô hình AgglomerativeClustering với 4 cụm, metric='euclidean', linkage='ward'
agg_cluster = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')

### TODO 2.3: Fit và gán nhãn dự báo cho X_scaled vào biến hierarchical_labels
hierarchical_labels = agg_cluster.fit_predict(X_scaled)

### TODO 2.4: Tính chỉ số Davies-Bouldin Index cho phân cụm thứ bậc
dbi_hierarchical = davies_bouldin_score(X_scaled, hierarchical_labels)

print(f"Chỉ số Davies-Bouldin Index của Hierarchical: {dbi_hierarchical:.4f}")
print("Đã hoàn thành Lab 2! Hãy chạy ô assert bên dưới để kiểm tra kết quả.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG CHO LAB 2 (ASSERT TEST)
assert Z_matrix.shape == (249, 4), "Lỗi: Ma trận liên kết cho 250 mẫu phải có kích thước (249, 4)!"
assert len(hierarchical_labels) == 250, "Lỗi: Số lượng nhãn phải bằng 250!"
assert len(set(hierarchical_labels)) == 4, "Lỗi: Số lượng cụm phải bằng 4!"
assert 0.0 < dbi_hierarchical < 1.5, "Lỗi: Chỉ số Davies-Bouldin không nằm trong khoảng kỳ vọng!"

print("CHÚC MỪNG! BẠN ĐÃ HOÀN THÀNH CHÍNH XÁC TẤT CẢ YÊU CẦU LAB 2!")


---

### Lab 3: DBSCAN & Tự động Lọc Nhiễu (Noise Detection)

#### Nhiệm vụ của bạn:
1. Dùng `NearestNeighbors(n_neighbors=5)` để tính khoảng cách tới láng giềng thứ 5 của mỗi mẫu trong `X_scaled`.
2. Khởi tạo mô hình `DBSCAN` với `eps=0.75` và `min_samples=5`.
3. Fit mô hình trên `X_scaled` và lưu nhãn vào biến `dbscan_labels`.
4. Xác định:
   - Số lượng cụm thực (không tính nhiễu `-1`). Lưu vào `n_clusters_dbscan`.
   - Số lượng điểm nhiễu (outliers mang nhãn `-1`). Lưu vào `n_noise_points`.



In [ ]:
# BÀI TẬP LAB 3: ĐIỀN CODE VÀO CÁC CHỖ TRỐNG TODO

### TODO 3.1: Khởi tạo NearestNeighbors với n_neighbors=5 và fit vào X_scaled
nn_model = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nn_model.kneighbors(X_scaled)

### TODO 3.2: Khởi tạo mô hình DBSCAN với eps=0.75 và min_samples=5
dbscan_model = DBSCAN(eps=0.75, min_samples=5)

### TODO 3.3: Fit và gán nhãn dự báo vào biến dbscan_labels
dbscan_labels = dbscan_model.fit_predict(X_scaled)

### TODO 3.4: Đếm số lượng cụm (loại trừ nhãn -1)
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)

### TODO 3.5: Đếm tổng số điểm nhiễu (nhãn -1)
n_noise_points = np.sum(dbscan_labels == -1)

print(f"Số lượng cụm phát hiện: {n_clusters_dbscan}")
print(f"Số lượng điểm nhiễu (Outliers): {n_noise_points}")
print("Đã hoàn thành Lab 3! Hãy chạy ô assert bên dưới để kiểm tra kết quả.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG CHO LAB 3 (ASSERT TEST)
assert len(dbscan_labels) == 250, "Lỗi: Số lượng nhãn phải bằng 250!"
assert n_clusters_dbscan >= 3, "Lỗi: DBSCAN phải tìm thấy ít nhất 3 cụm!"
assert n_noise_points >= 0, "Lỗi: Số điểm nhiễu không hợp lệ!"
assert -1 in dbscan_labels or n_noise_points == 0, "Lỗi cấu trúc nhãn DBSCAN!"

print("XUẤT SẮC! THUẬT TOÁN DBSCAN ĐÃ ĐƯỢC CẤU HÌNH VÀ THỰC THI CHUẨN XÁC!")


---

### Lab 4: Gaussian Mixture Models (GMM) & Lựa chọn Mô hình bằng BIC / AIC

#### Nhiệm vụ của bạn:
1. Viết vòng lặp thử nghiệm số thành phần $K \in [2, 8]$:
   - Fit mô hình `GaussianMixture(n_components=k, covariance_type='full', random_state=42)`.
   - Tính giá trị `.bic(X_scaled)` và lưu vào danh sách `bic_list`.
   - Tính giá trị `.aic(X_scaled)` và lưu vào danh sách `aic_list`.
2. Khởi tạo mô hình GMM tối ưu với $K = 4$ cụm và huấn luyện trên `X_scaled`.
3. Trích xuất ma trận xác suất thành viên mềm (Soft Assignment Probabilities) bằng phương thức `.predict_proba(X_scaled)`. Lưu vào biến `soft_probs`.
4. Trích xuất nhãn cứng phân cụm bằng phương thức `.predict(X_scaled)`. Lưu vào biến `gmm_labels`.



In [ ]:
# BÀI TẬP LAB 4: ĐIỀN CODE VÀO CÁC CHỖ TRỐNG TODO

bic_list = []
aic_list = []
k_test_range = range(2, 9)

for k in k_test_range:
    ### TODO 4.1: Khởi tạo mô hình GaussianMixture với n_components=k, covariance_type='full', random_state=42
    gmm_k = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    
    ### TODO 4.2: Fit mô hình gmm_k vào X_scaled
    gmm_k.fit(X_scaled)
    
    ### TODO 4.3: Tính giá trị bic và aic của gmm_k trên X_scaled
    bic_list.append(gmm_k.bic(X_scaled))
    aic_list.append(gmm_k.aic(X_scaled))

# Bước 2: Huấn luyện GMM với K=4 thành phần
### TODO 4.4: Khởi tạo GaussianMixture tối ưu với 4 thành phần
best_gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
best_gmm.fit(X_scaled)

### TODO 4.5: Trích xuất ma trận xác suất thành viên mềm
soft_probs = best_gmm.predict_proba(X_scaled)

### TODO 4.6: Trích xuất nhãn cụm cứng
gmm_labels = best_gmm.predict(X_scaled)

print("Đã hoàn thành Lab 4! Hãy chạy ô assert bên dưới để kiểm tra kết quả.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG CHO LAB 4 (ASSERT TEST)
assert len(bic_list) == 7 and len(aic_list) == 7, "Lỗi: Danh sách BIC/AIC phải có 7 phần tử!"
assert soft_probs.shape == (250, 4), "Lỗi: Ma trận xác suất mềm phải có kích thước (250, 4)!"
# Tổng xác suất của mỗi khách hàng trên 4 cụm phải bằng chính xác 1.0
assert np.allclose(np.sum(soft_probs, axis=1), 1.0), "Lỗi: Tổng xác suất các thành viên của mỗi điểm phải bằng 1.0!"
assert len(set(gmm_labels)) == 4, "Lỗi: GMM phải dự báo 4 cụm!"

print("HOÀN HẢO! BẠN ĐÃ LÀM CHỦ MÔ HÌNH XÁC SUẤT TRỘN GAUSSIAN VÀ THUẬT TOÁN EM!")


---

### Lab 5: Bảng Xếp hạng và Đánh giá So sánh Tổng hợp các Thuật toán

Cuối cùng, hãy tổng hợp chất lượng của 4 thuật toán trên dữ liệu khách hàng theo các chỉ số:
- **Silhouette Score** (Càng lớn càng tốt)
- **Davies-Bouldin Index** (Càng nhỏ càng tốt)
- **Calinski-Harabasz Index** (Càng lớn càng tốt)



In [ ]:
# Lọc bỏ nhiễu DBSCAN khi tính các chỉ số phân cụm nội bộ
valid_mask = dbscan_labels != -1

# Tính toán các chỉ số cho từng thuật toán
models_comparison = {
    "Thuật toán": ["K-Means (Centroid)", "Hierarchical (Ward)", "DBSCAN (Density)", "GMM (Distribution)"],
    "Silhouette Score (↑)": [
        silhouette_score(X_scaled, kmeans_labels),
        silhouette_score(X_scaled, hierarchical_labels),
        silhouette_score(X_scaled[valid_mask], dbscan_labels[valid_mask]),
        silhouette_score(X_scaled, gmm_labels)
    ],
    "Davies-Bouldin Index (↓)": [
        davies_bouldin_score(X_scaled, kmeans_labels),
        davies_bouldin_score(X_scaled, hierarchical_labels),
        davies_bouldin_score(X_scaled[valid_mask], dbscan_labels[valid_mask]),
        davies_bouldin_score(X_scaled, gmm_labels)
    ],
    "Calinski-Harabasz Index (↑)": [
        calinski_harabasz_score(X_scaled, kmeans_labels),
        calinski_harabasz_score(X_scaled, hierarchical_labels),
        calinski_harabasz_score(X_scaled[valid_mask], dbscan_labels[valid_mask]),
        calinski_harabasz_score(X_scaled, gmm_labels)
    ]
}

df_benchmark = pd.DataFrame(models_comparison)
df_benchmark.set_index("Thuật toán", inplace=True)
print("--- BẢNG SO SÁNH CHẤT LƯỢNG PHÂN CỤM DỮ LIỆU KHÁCH HÀNG ---")
df_benchmark.round(4)


## 5. Tổng kết & Đúc rút Kinh nghiệm Thực tiễn

Qua bài thực hành toàn diện này, chúng ta đã đúc rút được các nguyên tắc thực tế quan trọng:
1. **Chuẩn hóa dữ liệu (Feature Scaling) là bắt buộc:** Hầu hết các thuật toán phân cụm đều phụ thuộc vào khoảng cách Euclidean hoặc tích vô hướng. Nếu không chuẩn hóa, các biến có thang đo lớn (như *Annual_Income_k$*) sẽ lấn át hoàn toàn các biến còn lại.
2. **K-Means** phù hợp nhất khi cần tốc độ cao trên dữ liệu lớn và các cụm có hình cầu tách biệt đều đặn.
3. **Hierarchical Clustering** cho cái nhìn phả hệ sâu sắc thông qua Dendrogram, rất giá trị trong bài toán phân loại sinh học hoặc phân cấp phòng ban, nhóm sản phẩm.
4. **DBSCAN** là vũ khí hàng đầu khi dữ liệu chứa nhiều outlier, nhiễu và cụm có hình dạng cong xoắn phi tuyến tính.
5. **GMM** cung cấp độ tin cậy xác suất mềm, phù hợp cho các bài toán phân đoạn khách hàng đa chiều mà một người có thể mang đặc tính của nhiều phân khúc cùng lúc.

---
**Chúc mừng bạn đã hoàn thành xuất sắc bài thực hành Phân cụm Dữ liệu với Python!**

